# Fine-tuning Paraphrase Multilingual MPNet for Burmese Headline Generation

This notebook fine-tunes the `paraphrase-multilingual-mpnet-base-v2` model for generating Burmese news headlines from article text.

**Note**: Since MPNet is an encoder-only model, we'll adapt it for generation by either:
1. Using it as an encoder in an encoder-decoder architecture
2. Fine-tuning a separate seq2seq model with MPNet embeddings

We'll use approach 2 with mT5 for better multilingual support.

## 1. Setup and Installation

In [1]:
# Install required packages
!pip install -q transformers datasets sentence-transformers accelerate evaluate rouge-score sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.3 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from sentence_transformers import SentenceTransformer
import evaluate
from google.colab import drive

drive.mount('/content/drive')

In [2]:
# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## 2. Prepare Burmese Dataset

You'll need a dataset with Burmese articles and headlines. Here are some options:
- Use existing Burmese news datasets
- Load from CSV/JSON files
- Scrape Burmese news websites (with permission)

Expected format: `{'article': 'article text...', 'headline': 'headline text...'}`

In [3]:
df = pd.read_csv('/content/drive/MyDrive/NLP Project/Headline Generator Dataset/headline_corpus.csv')

In [4]:
# Split data into train/validation/test
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Convert to Hugging Face Dataset
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df)
})

print(dataset)

Train: 13547, Val: 1693, Test: 1694
DatasetDict({
    train: Dataset({
        features: ['id', 'headline', 'text', '__index_level_0__'],
        num_rows: 13547
    })
    validation: Dataset({
        features: ['id', 'headline', 'text', '__index_level_0__'],
        num_rows: 1693
    })
    test: Dataset({
        features: ['id', 'headline', 'text', '__index_level_0__'],
        num_rows: 1694
    })
})


In [5]:
print(df.head(3))

   id                                           headline  \
0   1  မော်လ်တာကမ်းလွန်၌ လှေမှောက်မှု ရွှေ့ပြောင်းနေထ...   
1   2  ၁၀ နှစ်ကြာ လုံခြုံရေးပူးပေါင်းဆောင်ရွက်မှု သဘေ...   
2   3  စစ်ပွဲလွန်ဂါဇာ၏ လုံခြုံရေးနှင့်အရပ်ဘက်ရေးရာမျာ...   

                                                text  
0  မော်လ်တာကမ်းလွန်မှာ တိမ်းမှောက်သွားတဲ့လှေကို ဖ...  
1  ဒိန်းမတ်နိုင်ငံဟာ ယူကရိန်းနဲ့ ၁၀ နှစ်ကြာ လုံခြ...  
2  စစ်ပွဲလွန်ဂါဇာကမ်းမြောင်ဒေသရဲ့ လုံခြုံရေးနဲ့ အ...  


## 3. Load Model and Tokenizer

We'll use mT5 (multilingual T5) which supports Burmese well for seq2seq tasks.

In [37]:
# Use mT5 for better multilingual support including Burmese
model_name = "google/mt5-small"  # Options: mt5-small, mt5-base, mt5-large

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"Model loaded: {model_name}")
print(f"Model parameters: {model.num_parameters():,}")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded: google/mt5-small
Model parameters: 556,291,456


## 4. Preprocess Data

In [38]:
import torch

def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["text"]]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        text_target=examples["headline"],
        max_length=64,
        truncation=True,
        padding="max_length",
    )

    labels_ids = [
        [(lid if lid != tokenizer.pad_token_id else -100) for lid in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = torch.tensor(labels_ids, dtype=torch.long)  # ✅ critical
    return model_inputs


# Apply preprocessing
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Tokenization complete!")
print(tokenized_dataset)

Map:   0%|          | 0/13547 [00:00<?, ? examples/s]

Map:   0%|          | 0/1693 [00:00<?, ? examples/s]

Map:   0%|          | 0/1694 [00:00<?, ? examples/s]

Tokenization complete!
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 13547
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1693
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1694
    })
})


In [39]:
sample = tokenized_dataset["train"][0]
print(sample["labels"][:20])


[259, 24122, 92894, 98238, 264, 3805, 121528, 97177, 98139, 95933, 259, 161086, 155726, 158580, 136137, 69855, 1, -100, -100, -100]


## 5. Training Setup

In [40]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Replace -100 so labels can be decoded
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(
        predictions, skip_special_tokens=True
    )
    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=True
    )

    # Compute ROUGE (no stemmer for Burmese)
    scores = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    # Convert to percentages
    return {
        "rouge1": round(scores["rouge1"] * 100, 4),
        "rouge2": round(scores["rouge2"] * 100, 4),
        "rougeL": round(scores["rougeL"] * 100, 4),
        "rougeLsum": round(scores["rougeLsum"] * 100, 4),
    }


In [68]:
# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./burmese-headline-generation",
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=2,       # can try 2 or 4
    per_device_eval_batch_size=1,        # small for fp16 safe validation
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,                           # keep for memory efficiency
    logging_dir="./logs",
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rouge1",
    generation_max_length=64,            # prevents OOM / NaN
    max_grad_norm=1.0,
    push_to_hub=False,
    report_to="none"
)


print("Training arguments configured")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments configured


In [69]:
# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

In [70]:
# Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer initialized and ready!")

Trainer initialized and ready!


In [74]:
batch = next(iter(trainer.get_train_dataloader()))
print(batch["labels"].dtype)  # should be torch.long
print(batch["labels"].shape)  # [batch_size, seq_len]
print(batch["labels"][0][:20])


torch.int64
torch.Size([2, 64])
tensor([   259,   1975,  34847,  48849, 189417, 111489,  67859,    261, 113780,
         27063,  17381,  21987,    259, 187505, 127566,  66924,   9410,  23569,
         34979,  34979], device='cuda:0')


In [75]:
for step, batch in enumerate(trainer.get_train_dataloader()):
    outputs = model(
        input_ids=batch["input_ids"].to(device),
        attention_mask=batch["attention_mask"].to(device),
        labels=batch["labels"].to(device)
    )
    print("Step", step, "Loss:", outputs.loss.item())
    if step > 5:
        break


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 351856 has 14.74 GiB memory in use. Of the allocated memory 14.57 GiB is allocated by PyTorch, and 35.35 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [48]:
print(tokenized_dataset["train"][0]["labels"][:20])
print(tokenized_dataset["train"].column_names)
batch = next(iter(trainer.get_train_dataloader()))
print(batch["labels"].shape)
print(batch["labels"][0][:20])


[259, 24122, 92894, 98238, 264, 3805, 121528, 97177, 98139, 95933, 259, 161086, 155726, 158580, 136137, 69855, 1, -100, -100, -100]
['input_ids', 'attention_mask', 'labels']
torch.Size([4, 64])
tensor([   259,   1975,  34847,  48849, 189417, 111489,  67859,    261, 113780,
         27063,  17381,  21987,    259, 187505, 127566,  66924,   9410,  23569,
         34979,  34979], device='cuda:0')


## 6. Train the Model

In [49]:
# Start training
print("Starting training...")
trainer.train()
print("Training complete!")

Starting training...


OutOfMemoryError: CUDA out of memory. Tried to allocate 490.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 286.12 MiB is free. Process 351856 has 14.46 GiB memory in use. Of the allocated memory 13.47 GiB is allocated by PyTorch, and 869.74 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 7. Evaluate the Model

In [44]:
# Evaluate on test set
eval_results = trainer.evaluate(tokenized_dataset["test"])
print("\nEvaluation Results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.000000,nan,0.000000,0.000000,0.000000,0.000000
2,0.000000,nan,0.000000,0.000000,0.000000,0.000000



Evaluation Results:
eval_loss: nan
eval_rouge1: 0.0000
eval_rouge2: 0.0000
eval_rougeL: 0.0000
eval_rougeLsum: 0.0000


## 8. Test Headline Generation

In [45]:
def generate_headline(article_text, max_length=128, num_beams=4):
    """Generate headline from article text"""
    # Prepare input
    input_text = "summarize: " + article_text
    inputs = tokenizer(
        input_text,
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # Generate
    model.to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=num_beams,
        early_stopping=True,
        no_repeat_ngram_size=3,
        length_penalty=1.0
    )

    # Decode
    headline = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return headline

In [47]:
# Test with examples from test set
test_examples = dataset["test"].select(range(min(3, len(dataset["test"]))))

print("=" * 80)
print("HEADLINE GENERATION EXAMPLES")
print("=" * 80)

for idx, example in enumerate(test_examples):
    article = example["text"]
    true_headline = example["headline"]
    generated_headline = generate_headline(article)

    print(f"\nExample {idx + 1}:")
    print("-" * 80)
    print(f"Article: {article[:200]}..." if len(article) > 200 else f"Article: {article}")
    print(f"\nTrue Headline: {true_headline}")
    print(f"Generated: {generated_headline}")
    print("-" * 80)

HEADLINE GENERATION EXAMPLES

Example 1:
--------------------------------------------------------------------------------
Article: အိန္ဒိယနိုင်ငံရဲ့ ၇၅ နှစ်မြောက် သမ္မတနိုင်ငံနေ့ကို နယူးဒေလီမြို့တော်မှာ ဇန်နဝါရီ ၂၆ ရက်က ကျင်းပခဲ့ပါတယ်။ နိုင်ငံရဲ့ စစ်ရေးစွမ်းရည်၊ အစဉ်အလာနဲ့ ယဉ်ကျေးမှုတို့ကို ပြသ ချီတက်ခဲ့တာပါ။ ချီတက်ပွဲမတိုင်ခင် အ...

True Headline: ၇၅ နှစ်မြောက် အိန္ဒိယသမ္မတ နိုင်ငံနေ့ နယူးဒေလီတွင် ကျင်းပ
Generated: <0x03>
--------------------------------------------------------------------------------

Example 2:
--------------------------------------------------------------------------------
Article: မန္တလေးတိုင်းဒေသကြီး မြင်းခြံခရိုင် ၄ မြို့နယ်မှာ   ဇူလိုင် ၃၀ ရက်ထိ  ကိုဗစ်-၁၉ အတည်ပြုလူနာ ၂၉၀၆ ဦးနဲ့ သေဆုံး ၁၀၇ ဦးရှိပြီလို့ ဒေသတွင်း ပရဟိတအသင်းတွေဆီက သိရပါတယ်။ မြင်းခံမြို့နယ်မှာ အတည်ပြုလူနာ ၅၃၂ ဦး...

True Headline: မြင်းခြံခရိုင်တွင် ကိုဗစ်အတည်ပြုလူနာ ၂၉၀၀ ကျော်နှင့် သေဆုံးသူ ၁၀၇ ဦးရှိလာ
Generated: <0x03>
------------------------------------------------------------------------------

In [48]:
# Interactive testing - Try your own Burmese text
custom_article = """
ရန်ကုန်မြို့ရှိ ဈေးကွက်များတွင် ဒီဇင်ဘာလအတွင်း
စားသောက်ကုန်ဈေးနှုန်းများ သိသိသာသာ မြင့်တက်လာခဲ့ပါသည်။
"""

print("Custom Article:")
print(custom_article)
print("\nGenerated Headline:")
print(generate_headline(custom_article.strip()))

Custom Article:

ရန်ကုန်မြို့ရှိ ဈေးကွက်များတွင် ဒီဇင်ဘာလအတွင်း 
စားသောက်ကုန်ဈေးနှုန်းများ သိသိသာသာ မြင့်တက်လာခဲ့ပါသည်။


Generated Headline:
<0x03>


## 9. Save the Model

In [ ]:
# Save model locally
output_dir = "./burmese-headline-model-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

Optional: Save to Google Drive
!cp -r {output_dir} /content/drive/MyDrive/
print("Model copied to Google Drive")

## 10. Load Saved Model (for future use)

In [ ]:
# Load the saved model
# loaded_model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
# loaded_tokenizer = AutoTokenizer.from_pretrained(output_dir)
# print("Model loaded successfully!")

## 11. Export for Production (Optional)

In [ ]:
# Optional: Convert to ONNX for faster inference
# !pip install -q optimum[exporters]

# from optimum.onnxruntime import ORTModelForSeq2SeqLM

# ort_model = ORTModelForSeq2SeqLM.from_pretrained(
#     output_dir,
#     export=True
# )
# ort_model.save_pretrained("./burmese-headline-onnx")
# print("ONNX model exported")

## Notes and Tips

### Improving Performance:
1. **More Data**: Collect more Burmese news articles with headlines (1000+ examples recommended)
2. **Larger Model**: Try `google/mt5-base` or `google/mt5-large` for better quality
3. **Data Augmentation**: Back-translation or paraphrasing of existing data
4. **Hyperparameter Tuning**: Adjust learning rate, batch size, num_beams
5. **Preprocessing**: Clean and normalize Burmese text properly

### Model Options:
- `google/mt5-small`: Fast, good for prototyping (~300M params)
- `google/mt5-base`: Better quality (~580M params)
- `google/mt5-large`: Best quality (~1.2B params, requires more GPU)

### Dataset Sources:
- Burmese news websites
- Myanmar Wikipedia articles
- Public Burmese NLP datasets

### GPU Memory Tips:
- Reduce `per_device_train_batch_size` if OOM error
- Use gradient accumulation: `gradient_accumulation_steps=2`
- Enable `fp16=True` for mixed precision training
